In [1]:
from chembl_webresource_client.new_client import new_client
import pandas as pd


In [2]:
molecule = new_client.molecule
target = new_client.target
activity = new_client.activity

In [3]:
res = molecule.filter(pref_name__iexact="IMATINIB")
for r in res:
    print(r['molecule_chembl_id'], r['pref_name'])

CHEMBL941 IMATINIB


In [4]:
acts = activity.filter(molecule_chembl_id="CHEMBL941").filter(pchembl_value__isnull=False)
df = pd.DataFrame(list(acts))
df[['molecule_chembl_id','target_chembl_id','standard_type','standard_relation',
    'standard_value','standard_units','pchembl_value']].head(20)

,molecule_chembl_id,target_chembl_id,standard_type,standard_relation,standard_value,standard_units,pchembl_value
0,CHEMBL941,CHEMBL2111414,IC50,=,40.0,nM,7.40
1,CHEMBL941,CHEMBL385,IC50,=,110.0,nM,6.96
2,CHEMBL941,CHEMBL385,IC50,=,400.0,nM,6.40
3,CHEMBL941,CHEMBL1913,IC50,=,240.0,nM,6.62
4,CHEMBL941,CHEMBL1936,IC50,=,260.0,nM,6.58
5,CHEMBL941,CHEMBL1844,IC50,=,960.0,nM,6.02
6,CHEMBL941,CHEMBL1862,IC50,=,38.0,nM,7.42
7,CHEMBL941,CHEMBL2095189,IC50,=,50.0,nM,7.30
8,CHEMBL941,CHEMBL1936,IC50,=,100.0,nM,7.00
9,CHEMBL941,CHEMBL1936,IC50,=,10000.0,nM,5.00


In [5]:
tgt = target.filter(target_synonym__icontains="ABL1", organism="Homo sapiens")
for t in tgt:
    print(t['target_chembl_id'], t['pref_name'], t['organism'])

CHEMBL1862 Tyrosine-protein kinase ABL1 Homo sapiens
CHEMBL2096618 Bcr/Abl fusion protein Homo sapiens
CHEMBL2111414 Tyrosine-protein kinase ABL Homo sapiens
CHEMBL3885630 Ras and Rab interactor 1/Tyrosine-protein kinase ABL1 Homo sapiens
CHEMBL3885645 Transcription factor ETV6/Tyrosine-protein kinase ABL1 Homo sapiens
CHEMBL4296119 Baculoviral IAP repeat-containing protein 2/Tyrosine-protein kinase ABL1/Breakpoint cluster region protein Homo sapiens
CHEMBL4296120 E3 ubiquitin-protein ligase XIAP/Tyrosine-protein kinase ABL1/Breakpoint cluster region protein Homo sapiens
CHEMBL4296137 Protein cereblon/Tyrosine-protein kinase ABL1/Breakpoint cluster region protein Homo sapiens
CHEMBL4523725 von Hippel-Lindau disease tumor suppressor/Tyrosine-protein kinase ABL1 Homo sapiens
CHEMBL4523750 Protein cereblon/Tyrosine-protein kinase ABL1 Homo sapiens
CHEMBL4523751 von Hippel-Lindau disease tumor suppressor/Tyrosine-protein kinase ABL1/Breakpoint cluster region protein Homo sapiens
CHEMBL6067

## What I observed (filled in 2026-09-27 from the executed cells below)

Imatinib resolved to CHEMBL941. With the `pchembl_value__isnull=False` filter the activity query
returns **833** rows; **unfiltered it returns 4,878**. The filter therefore hides 83% of the record,
including **every** censored row: all 1,335 rows with relation `>`, `<`, `>=`, `<=` have a null pChEMBL.

Concrete censored example: **activity_id 884645** (assay CHEMBL766072) — standard_type=IC50,
standard_relation='>', standard_value=30000, standard_units='nM' against **CHEMBL1974 (FLT3, human)**.
This is a censored measurement: the assay stopped at 30 µM without finding where activity falls off —
it is *not* a potency of exactly 30,000 nM. (Similar: activity 909964, IC50 > 100,000 nM vs EGFR, CHEMBL203.)

Relation counts for imatinib (unfiltered): `=` 2,683 · `>` 1,323 · `<` 6 · `>=` 6 · `~` 1 · missing 859.
The 859 missing-relation rows are mostly non-potency endpoints (e.g. drug degradation, solubility),
so "censored" must be tested with an explicit operator list, not `!= '='` (which also catches NaN).

ABL1 (human) resolved to target **CHEMBL1862** (Tyrosine-protein kinase ABL1, Homo sapiens).
Note: CHEMBL2111414 ("Tyrosine-protein kinase ABL") and several fusion-protein targets also match the
synonym search — only single-protein, human targets are in scope (Briefing §3.2).

In [6]:
acts_all = activity.filter(molecule_chembl_id="CHEMBL941")
df_all = pd.DataFrame(list(acts_all))
print(f"Total rows without the pchembl filter: {len(df_all)}")
df_all['standard_relation'].value_counts()

Total rows without the pchembl filter: 4878


standard_relation
=     2683
>     1323
<        6
>=       6
~        1
Name: count, dtype: int64

In [7]:
# explicit operator list: `!= '='` would also catch the NaN-relation (non-potency) rows
censored = df_all[df_all['standard_relation'].isin(['>', '<', '>=', '<='])]
print(f"censored rows: {len(censored)}, of which pchembl null: {censored['pchembl_value'].isna().sum()}")
censored[['molecule_chembl_id','target_chembl_id','standard_type','standard_relation',
          'standard_value','standard_units','pchembl_value']]

censored rows: 1335, of which pchembl null: 1335


,molecule_chembl_id,target_chembl_id,standard_type,standard_relation,standard_value,standard_units,pchembl_value
6,CHEMBL941,CHEMBL1974,IC50,>,30000.0,nM,NaN
9,CHEMBL941,CHEMBL203,IC50,>,100000.0,nM,NaN
10,CHEMBL941,CHEMBL267,IC50,>,100000.0,nM,NaN
12,CHEMBL941,CHEMBL2094138,IC50,>,500000.0,nM,NaN
13,CHEMBL941,CHEMBL299,IC50,>,100000.0,nM,NaN
...,...,...,...,...,...,...,...
4828,CHEMBL941,CHEMBL2996,IC50,>,100000.0,nM,NaN
4859,CHEMBL941,CHEMBL2362975,Solubility,>,122.0,ug.mL-1,NaN
4868,CHEMBL941,CHEMBL1936,IC50,>=,10000.0,nM,NaN
4871,CHEMBL941,CHEMBL2007,IC50,>=,10000.0,nM,NaN
